v2 model:
- Filter train/val departures by [1, 99]-th percentiles (around 5 and 40 minutes)
- Use last available month (Dec. 2025) as a validation set
- Arrival movements are not used
- Used features:
    - FLIGHT_RULE_mvt
    - ADEP_mvt
    - AIRCRAFT_TYPE_mvt
    - RUNWAY_mvt
    - STAND_mvt
    - MARKET_SEGMENT_flt
    - FLIGHT_RULE_flt
    - FLIGHT_TYPE_flt
    - AIRCRAFT_TYPE_flt
    - WK_TBL_CAT_flt
    - Month/day/dow/hour extracted from datetime cols like MVT_TIME_UTC_mvt, etc.
- Target: TAXITIME_SEC_mvt
- Model: CatBoost

In [1]:
import os

import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostRegressor

/opt/miniconda3/envs/ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read training data

In [ ]:
def read_train(data_dir: str) -> pd.DataFrame:
    filenames = [x for x in os.listdir(data_dir) if x.startswith("training")]
    dfs = []
    for filename in filenames:
        month = int(filename.split("_")[1].split("-")[1])
        df = pd.read_parquet(os.path.join(data_dir, filename))
        df["month"] = month
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [3]:
df = read_train(data_dir="../data")
df.shape

(4167797, 31)

### Drop arrivals

In [4]:
df = df[df["PHASE_mvt"] == "DEP"]
df.shape

(2085047, 31)

### Extract month/day/dow/hour from timestamp cols, apply sin-cos encoding

In [5]:
timestamp_cols = [
    "MVT_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
    "LOBT_flt",
    "IOBT_flt",
    "EOBT_1_flt",
    "ARVT_1_flt",
    "AOBT_3_flt",
    "ARVT_3_flt"
]
timestamp_features = []
for col in timestamp_cols:
    print(f"Processing: {col}")

    month = df[col].dt.month
    day = df[col].dt.day
    dow = df[col].dt.day_of_week
    hour = df[col].dt.hour

    # Month
    df[f"{col}_month_sin"] = np.sin(2 * np.pi * month / 12)
    df[f"{col}_month_cos"] = np.cos(2 * np.pi * month / 12)
    # Day
    df[f"{col}_day_sin"] = np.sin(2 * np.pi * day / 31)
    df[f"{col}_day_cos"] = np.cos(2 * np.pi * day / 31)
    # Day of week
    df[f"{col}_dow_sin"] = np.sin(2 * np.pi * dow / 7)
    df[f"{col}_dow_cos"] = np.cos(2 * np.pi * dow / 7)
    # Hour
    df[f"{col}_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df[f"{col}_hour_cos"] = np.cos(2 * np.pi * hour / 24)


    timestamp_features.extend([
        f"{col}_month_sin",
        f"{col}_month_cos",
        f"{col}_day_sin",
        f"{col}_day_cos",
        f"{col}_dow_sin",
        f"{col}_dow_cos",
        f"{col}_hour_sin",
        f"{col}_hour_cos"
    ])

Processing: MVT_TIME_UTC_mvt
Processing: SCHED_TIME_UTC_mvt
Processing: LOBT_flt
Processing: IOBT_flt
Processing: EOBT_1_flt
Processing: ARVT_1_flt
Processing: AOBT_3_flt
Processing: ARVT_3_flt


### Features/target

In [6]:
cat_features = [
    "FLIGHT_RULE_mvt",
    "ADEP_mvt",
    "AIRCRAFT_TYPE_mvt",
    "RUNWAY_mvt",
    "STAND_mvt",
    "MARKET_SEGMENT_flt",
    "FLIGHT_RULE_flt",
    "FLIGHT_TYPE_flt",
    "AIRCRAFT_TYPE_flt",
    "WK_TBL_CAT_flt"
]
target = "TAXITIME_SEC_mvt"
cat_features

['FLIGHT_RULE_mvt',
 'ADEP_mvt',
 'AIRCRAFT_TYPE_mvt',
 'RUNWAY_mvt',
 'STAND_mvt',
 'MARKET_SEGMENT_flt',
 'FLIGHT_RULE_flt',
 'FLIGHT_TYPE_flt',
 'AIRCRAFT_TYPE_flt',
 'WK_TBL_CAT_flt']

### Fill NaN values in categorical features

In [9]:
df[cat_features] = df[cat_features].fillna("NAN")
df[cat_features].isna().sum()

FLIGHT_RULE_mvt       0
ADEP_mvt              0
AIRCRAFT_TYPE_mvt     0
RUNWAY_mvt            0
STAND_mvt             0
MARKET_SEGMENT_flt    0
FLIGHT_RULE_flt       0
FLIGHT_TYPE_flt       0
AIRCRAFT_TYPE_flt     0
WK_TBL_CAT_flt        0
dtype: int64

### Split to train/val

In [10]:
df_train = df[df["month"] != 12]
df_val = df[df["month"] == 12]
df_train.shape, df_val.shape

((1919370, 95), (165677, 95))

### Drop train/val departures by [1, 99] range

In [11]:
q_1 = df_train["TAXITIME_SEC_mvt"].quantile(0.01)
q_99 = df_train["TAXITIME_SEC_mvt"].quantile(0.99)
df_train = df_train[(df_train["TAXITIME_SEC_mvt"] >= q_1) & (df_train["TAXITIME_SEC_mvt"] <= q_99)]
df_val = df_val[(df_val["TAXITIME_SEC_mvt"] >= q_1) & (df_val["TAXITIME_SEC_mvt"] <= q_99)]
print(f"1st percentile: {q_1}, 99th percentile: {q_99}")
df_train.shape, df_val.shape

1st percentile: 291.0, 99th percentile: 2339.0


((1881066, 95), (162569, 95))

### Hyperparameters tuning

In [12]:
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 10, 100),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "verbose": False,
    }

    model = CatBoostRegressor(**params, cat_features=cat_features)
    model.fit(
        X=df_train[cat_features + timestamp_features],
        y=df_train[target],
        eval_set=(df_val[cat_features + timestamp_features], df_val[target]),
        early_stopping_rounds=50,
    )
    train_preds = model.predict(df_train[cat_features + timestamp_features])
    train_rmse = ((train_preds - df_train[target]) ** 2).mean() ** 0.5
    val_preds = model.predict(df_val[cat_features + timestamp_features])
    val_rmse = ((val_preds - df_val[target]) ** 2).mean() ** 0.5
    print(f"Trial {trial.number}: train RMSE = {train_rmse}, val RMSE = {val_rmse}")

    return val_rmse


In [13]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)
study.best_params

[I 2026-10-06 05:32:52,394] A new study created in memory with name: no-name-29ebee3b-45bd-4157-9a15-8ffdc0472bf0
[I 2026-10-06 05:32:59,335] Trial 0 finished with value: 350.31319174419974 and parameters: {'iterations': 10, 'depth': 4, 'learning_rate': 0.001529366256110671, 'l2_leaf_reg': 0.029911366671405374, 'random_strength': 0.24117594486528454, 'bagging_temperature': 0.7837979818834188, 'border_count': 42}. Best is trial 0 with value: 350.31319174419974.


Trial 0: train RMSE = 346.2142653740819, val RMSE = 350.31319174419974


[I 2026-10-06 05:33:20,130] Trial 1 finished with value: 341.9038925783665 and parameters: {'iterations': 90, 'depth': 4, 'learning_rate': 0.0013044340969207099, 'l2_leaf_reg': 4.329722657174614, 'random_strength': 0.007006592443613063, 'bagging_temperature': 0.6480393745380716, 'border_count': 239}. Best is trial 1 with value: 341.9038925783665.


Trial 1: train RMSE = 337.2759875313873, val RMSE = 341.9038925783665


[I 2026-10-06 05:33:37,442] Trial 2 finished with value: 277.7314318515632 and parameters: {'iterations': 47, 'depth': 7, 'learning_rate': 0.09124171450273515, 'l2_leaf_reg': 1.6928025958107387, 'random_strength': 0.002177965135544397, 'bagging_temperature': 0.31455240885831837, 'border_count': 144}. Best is trial 2 with value: 277.7314318515632.


...

[I 2026-10-06 06:15:24,965] Trial 48 finished with value: 265.08056766019627 and parameters: {'iterations': 78, 'depth': 10, 'learning_rate': 0.09795126699802825, 'l2_leaf_reg': 0.0025032445693140815, 'random_strength': 0.0076641062432915, 'bagging_temperature': 0.045820479102906105, 'border_count': 92}. Best is trial 48 with value: 265.08056766019627.


Trial 48: train RMSE = 248.33441983136282, val RMSE = 265.08056766019627


[I 2026-10-06 06:16:31,471] Trial 49 finished with value: 265.07351201129455 and parameters: {'iterations': 90, 'depth': 10, 'learning_rate': 0.08557783034150712, 'l2_leaf_reg': 0.005774041153176739, 'random_strength': 0.058583567201748094, 'bagging_temperature': 0.03541649280101082, 'border_count': 73}. Best is trial 49 with value: 265.07351201129455.


Trial 49: train RMSE = 248.450255115581, val RMSE = 265.07351201129455


{'iterations': 90,
 'depth': 10,
 'learning_rate': 0.08557783034150712,
 'l2_leaf_reg': 0.005774041153176739,
 'random_strength': 0.058583567201748094,
 'bagging_temperature': 0.03541649280101082,
 'border_count': 73}

### Refit on full data

In [14]:
df = df[(df["TAXITIME_SEC_mvt"] >= q_1) & (df["TAXITIME_SEC_mvt"] <= q_99)]
submit_model = CatBoostRegressor(**study.best_params, cat_features=cat_features)
submit_model.fit(
    X=df[cat_features + timestamp_features],
    y=df[target],
    early_stopping_rounds=50,
)

0:	learn: 338.7094015	total: 760ms	remaining: 1m 7s
1:	learn: 330.5419920	total: 1.54s	remaining: 1m 7s
2:	learn: 323.4239530	total: 2.1s	remaining: 1m
3:	learn: 317.2940140	total: 2.59s	remaining: 55.6s
4:	learn: 312.1045357	total: 3.19s	remaining: 54.1s
5:	learn: 307.3859701	total: 3.78s	remaining: 53s
6:	learn: 303.1724625	total: 4.24s	remaining: 50.3s
7:	learn: 299.6031855	total: 4.71s	remaining: 48.3s
8:	learn: 296.5080940	total: 5.3s	remaining: 47.7s
9:	learn: 291.9774066	total: 5.89s	remaining: 47.1s
10:	learn: 288.1394225	total: 6.36s	remaining: 45.7s
11:	learn: 284.9189576	total: 6.94s	remaining: 45.1s
12:	learn: 282.1737211	total: 7.53s	remaining: 44.6s
13:	learn: 279.6931145	total: 8.12s	remaining: 44.1s
14:	learn: 277.6407280	total: 8.7s	remaining: 43.5s
15:	learn: 275.6426473	total: 9.17s	remaining: 42.4s
16:	learn: 273.9539393	total: 9.79s	remaining: 42s
17:	learn: 272.5452937	total: 10.4s	remaining: 41.6s
18:	learn: 271.3053681	total: 11s	remaining: 41.2s
19:	learn: 270.

CatBoostRegressor(bagging_temperature=0.03541649280101082, border_count=73, cat_features=['FLIGHT_RULE_mvt', 'ADEP_mvt', 'AIRCRAFT_TYPE_mvt', 'RUNWAY_mvt', 'STAND_mvt', 'MARKET_SEGMENT_flt', 'FLIGHT_RULE_flt', 'FLIGHT_TYPE_flt', 'AIRCRAFT_TYPE_flt', 'WK_TBL_CAT_flt'], depth=10, iterations=90, l2_leaf_reg=0.005774041153176739, learning_rate=0.08557783034150712, loss_function='RMSE', random_strength=0.058583567201748094)

### Features importances

In [15]:
importances = dict(zip(cat_features + timestamp_features, submit_model.feature_importances_))
dict(sorted(importances.items(), key=lambda item: item[1]))

{'FLIGHT_RULE_mvt': np.float64(0.0),
 'FLIGHT_RULE_flt': np.float64(0.0),
 'MVT_TIME_UTC_mvt_day_sin': np.float64(0.0),
 'MVT_TIME_UTC_mvt_day_cos': np.float64(0.0),
 'SCHED_TIME_UTC_mvt_dow_sin': np.float64(0.0),
 'LOBT_flt_month_sin': np.float64(0.0),
 'LOBT_flt_day_sin': np.float64(0.0),
 'LOBT_flt_day_cos': np.float64(0.0),
 'LOBT_flt_dow_cos': np.float64(0.0),
 'IOBT_flt_day_sin': np.float64(0.0),
 'IOBT_flt_day_cos': np.float64(0.0),
 'IOBT_flt_dow_cos': np.float64(0.0),
 'EOBT_1_flt_month_sin': np.float64(0.0),
 'EOBT_1_flt_day_sin': np.float64(0.0),
 'EOBT_1_flt_day_cos': np.float64(0.0),
 'EOBT_1_flt_dow_sin': np.float64(0.0),
 'EOBT_1_flt_dow_cos': np.float64(0.0),
 'ARVT_1_flt_day_sin': np.float64(0.0),
 'ARVT_1_flt_day_cos': np.float64(0.0),
 'ARVT_1_flt_dow_sin': np.float64(0.0),
 'ARVT_1_flt_dow_cos': np.float64(0.0),
 'AOBT_3_flt_day_sin': np.float64(0.0),
 'AOBT_3_flt_day_cos': np.float64(0.0),
 'AOBT_3_flt_dow_sin': np.float64(0.0),
 'AOBT_3_flt_dow_cos': np.float64(0.

### Make predictions for submit

In [17]:
df_test = pd.read_parquet("../data/ranking.parquet")
df_test = df_test[df_test["PHASE_mvt"] == "DEP"]
df_test.shape

(344841, 30)

In [18]:
df_test[cat_features] = df_test[cat_features].fillna("NAN")
df_test[cat_features].isna().sum()

FLIGHT_RULE_mvt       0
ADEP_mvt              0
AIRCRAFT_TYPE_mvt     0
RUNWAY_mvt            0
STAND_mvt             0
MARKET_SEGMENT_flt    0
FLIGHT_RULE_flt       0
FLIGHT_TYPE_flt       0
AIRCRAFT_TYPE_flt     0
WK_TBL_CAT_flt        0
dtype: int64

In [19]:
for col in timestamp_cols:
    print(f"Processing: {col}")

    month = df_test[col].dt.month
    day = df_test[col].dt.day
    dow = df_test[col].dt.day_of_week
    hour = df_test[col].dt.hour

    # Month
    df_test[f"{col}_month_sin"] = np.sin(2 * np.pi * month / 12)
    df_test[f"{col}_month_cos"] = np.cos(2 * np.pi * month / 12)
    # Day
    df_test[f"{col}_day_sin"] = np.sin(2 * np.pi * day / 31)
    df_test[f"{col}_day_cos"] = np.cos(2 * np.pi * day / 31)
    # Day of week
    df_test[f"{col}_dow_sin"] = np.sin(2 * np.pi * dow / 7)
    df_test[f"{col}_dow_cos"] = np.cos(2 * np.pi * dow / 7)
    # Hour
    df_test[f"{col}_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df_test[f"{col}_hour_cos"] = np.cos(2 * np.pi * hour / 24)

Processing: MVT_TIME_UTC_mvt
Processing: SCHED_TIME_UTC_mvt
Processing: LOBT_flt
Processing: IOBT_flt
Processing: EOBT_1_flt
Processing: ARVT_1_flt
Processing: AOBT_3_flt
Processing: ARVT_3_flt


In [20]:
test_preds = submit_model.predict(df_test[cat_features + timestamp_features])
test_preds[:5]

array([1080.51170804,  931.54054114,  926.47129448,  897.53025223,
        688.41061932])

In [21]:
submit = pd.DataFrame({
    "MVT_ID_mvt": df_test["MVT_ID_mvt"],
    "TAXITIME_SEC_mvt": test_preds
})
submit.to_parquet("incredible-rose_v2.parquet")
submit.shape

(344841, 2)